# 14 — Databases & Persistence

## 📓 Interactive Notebook · Module 10 · Advanced

In this notebook, you'll learn:
1. **Why databases** are essential for Data Science apps
2. **Relational database concepts** (tables, keys, relationships)
3. **SQL basics** (SELECT, INSERT, UPDATE, DELETE)
4. **Python + SQLite** connection and querying
5. **Parameterized queries** for security
6. **Streamlit dashboard** with database backend

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Connect Python to SQLite databases
- Execute SQL queries and display results
- Use parameterized queries to prevent SQL injection
- Build a Streamlit app with database backend
- Implement CRUD operations

## 📋 Prerequisites

- Modules 01–09 completed
- Basic understanding of data structures
- Pandas knowledge

---

## 💡 Why Databases?

### Problems with File-Based Data

| Problem | Impact |
|---------|--------|
| No concurrent access | Two users can't write simultaneously |
| No querying | Must load entire file to filter |
| No transactions | Partial writes corrupt data |
| No relationships | Data duplication across files |

### Database Benefits

| Benefit | Impact |
|---------|--------|
| Concurrent access | Multiple users read/write safely |
| SQL queries | Filter, aggregate, join efficiently |
| ACID transactions | Consistent, reliable writes |
| Relationships | Normalized data, no duplication |

---

## 💡 SQLite — The Teaching Database

SQLite is perfect for learning:
- **No server required** — database is a single file
- **Built into Python** — no installation needed
- **Focus on SQL** — not infrastructure

For production, use PostgreSQL or MySQL.

In [ ]:
import sqlite3
import pandas as pd
import os

st.header("💡 SQLite — The Teaching Database")

# Create an in-memory database for this notebook
conn = sqlite3.connect(":memory:")

# Create a table
conn.execute("""
    CREATE TABLE students (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE,
        score REAL,
        department TEXT
    )
""")

# Insert sample data
students_data = [
    ("Alice", "alice@university.edu", 95, "Computer Science"),
    ("Bob", "bob@university.edu", 87, "Mathematics"),
    ("Charlie", "charlie@university.edu", 92, "Computer Science"),
    ("Diana", "diana@university.edu", 78, "Physics"),
    ("Eve", "eve@university.edu", 96, "Mathematics"),
]

conn.executemany(
    "INSERT INTO students (name, email, score, department) VALUES (?, ?, ?, ?)",
    students_data
)
conn.commit()

st.success("✅ Database created with 5 students")

---

## 💡 Basic SQL Queries

### SELECT — Read Data

In [ ]:
import streamlit as st
import sqlite3
import pandas as pd

st.header("📝 SQL Basics: SELECT")

# Reuse the in-memory connection
# (In real apps, use @st.cache_resource)

# Select all
st.subheader("SELECT * FROM students")
df = pd.read_sql("SELECT * FROM students", conn)
st.dataframe(df)

# Select with condition
st.subheader("SELECT * FROM students WHERE score > 90")
df_high = pd.read_sql(
    "SELECT * FROM students WHERE score > 90",
    conn
)
st.dataframe(df_high)

# Ordering
st.subheader("ORDER BY score DESC")
df_sorted = pd.read_sql(
    "SELECT name, score FROM students ORDER BY score DESC",
    conn
)
st.dataframe(df_sorted)

### Aggregation

In [ ]:
import streamlit as st
import sqlite3
import pandas as pd

st.header("📊 Aggregation Queries")

# Aggregate functions
st.subheader("Aggregate Statistics")
stats = pd.read_sql("""
    SELECT 
        COUNT(*) as total_students,
        AVG(score) as avg_score,
        MIN(score) as min_score,
        MAX(score) as max_score
    FROM students
""", conn)
st.dataframe(stats)

# Group by
st.subheader("Average Score by Department")
dept_stats = pd.read_sql("""
    SELECT 
        department,
        COUNT(*) as student_count,
        ROUND(AVG(score), 1) as avg_score
    FROM students
    GROUP BY department
    ORDER BY avg_score DESC
""", conn)
st.dataframe(dept_stats)
st.bar_chart(dept_stats.set_index("department")["avg_score"])

---

## ⚠️ SQL Injection — Security Critical!

**NEVER** concatenate user input into SQL queries!

In [ ]:
import streamlit as st
import sqlite3

st.header("⚠️ SQL Injection Awareness")

st.subheader("❌ DANGEROUS: Never do this!")
st.code('''
# This is VULNERABLE to SQL injection!
user_input = st.text_input("Enter student name:")
query = f"SELECT * FROM students WHERE name = '{user_input}'"
cursor.execute(query)

# If user enters: \' OR \'1\' = \'1
# Query becomes: SELECT * FROM students WHERE name = \'\' OR \'1\' = \'1\'
# This returns ALL students!
''', language="python")

st.subheader("✅ SAFE: Use parameterized queries!")
st.code('''
# This is SAFE from SQL injection
user_input = st.text_input("Enter student name:")
query = "SELECT * FROM students WHERE name = ?"
cursor.execute(query, (user_input,))
# User input is treated as a VALUE, not SQL code
''', language="python")

# Demonstrate safe query
st.subheader("Safe Query Demo")
search_name = st.text_input("Search student by name:", "Alice")

# Parameterized query
result = pd.read_sql(
    "SELECT * FROM students WHERE name = ?",
    conn,
    params=(search_name,)
)
st.dataframe(result)

---

## 💡 CRUD Operations

Create, Read, Update, Delete — the four basic database operations.

In [ ]:
import streamlit as st
import sqlite3
import pandas as pd

st.header("🔄 CRUD Operations")

# CREATE
st.subheader("CREATE — Add New Student")
with st.form("add_student"):
    col1, col2 = st.columns(2)
    with col1:
        new_name = st.text_input("Name")
        new_email = st.text_input("Email")
    with col2:
        new_score = st.number_input("Score", 0, 100, 85)
        new_dept = st.selectbox("Department", ["Computer Science", "Mathematics", "Physics"])
    
    if st.form_submit_button("Add Student"):
        try:
            conn.execute(
                "INSERT INTO students (name, email, score, department) VALUES (?, ?, ?, ?)",
                (new_name, new_email, new_score, new_dept)
            )
            conn.commit()
            st.success(f"Added {new_name}!")
            st.rerun()
        except sqlite3.IntegrityError:
            st.error("Email already exists!")

# READ
st.subheader("READ — View Students")
df = pd.read_sql("SELECT * FROM students ORDER BY score DESC", conn)
st.dataframe(df, use_container_width=True)

# UPDATE
st.subheader("UPDATE — Modify Score")
with st.form("update_score"):
    student_to_update = st.selectbox("Select student", df["name"].tolist())
    new_score_val = st.number_input("New score", 0, 100, 90)
    
    if st.form_submit_button("Update Score"):
        conn.execute(
            "UPDATE students SET score = ? WHERE name = ?",
            (new_score_val, student_to_update)
        )
        conn.commit()
        st.success(f"Updated {student_to_update}'s score!")
        st.rerun()

# DELETE
st.subheader("DELETE — Remove Student")
student_to_delete = st.selectbox("Select student to delete", df["name"].tolist(), key="delete")
if st.button("Delete Student"):
    conn.execute("DELETE FROM students WHERE name = ?", (student_to_delete,))
    conn.commit()
    st.success(f"Deleted {student_to_delete}")
    st.rerun()

---

## 💡 Connection Caching with Streamlit

Use `@st.cache_resource` to cache database connections.

In [ ]:
import streamlit as st
import sqlite3
import pandas as pd

st.header("💾 Connection Caching Pattern")

st.code('''
import streamlit as st
import sqlite3

@st.cache_resource
def get_connection():
    """Cache the database connection as a singleton."""
    return sqlite3.connect("data.db", check_same_thread=False)

@st.cache_data(ttl=60)
def get_products():
    """Cache query results for 60 seconds."""
    conn = get_connection()
    return pd.read_sql("SELECT * FROM products", conn)

def add_product(name, price):
    conn = get_connection()
    conn.execute(
        "INSERT INTO products (name, price) VALUES (?, ?)",
        (name, price)
    )
    conn.commit()
    get_products.clear()  # Invalidate cache after write
''', language="python")

st.write("**Key points:**")
st.write("1. Use `@st.cache_resource` for connections (returns same object)")
st.write("2. Use `@st.cache_data` for query results (returns copies)")
st.write("3. Call `.clear()` after writes to invalidate cache")
st.write("4. Use `check_same_thread=False` for SQLite")

---

## 💡 Secrets Management

Never hard-code credentials. Use Streamlit secrets.

In [ ]:
import streamlit as st

st.header("🔐 Secrets Management")

st.subheader("For this notebook (SQLite), we use a file path:")
st.code('''
# SQLite — just a file path
import sqlite3
conn = sqlite3.connect("my_database.db")
''', language="python")

st.subheader("For production (PostgreSQL), use st.secrets:")
st.code('''
# .streamlit/secrets.toml (NEVER commit this file!)
# [database]
# host = "localhost"
# port = 5432
# name = "myapp_db"
# username = "admin"
# password = "secure_password"

# In your app:
import streamlit as st
import psycopg2

conn = psycopg2.connect(
    host=st.secrets["database"]["host"],
    port=st.secrets["database"]["port"],
    dbname=st.secrets["database"]["name"],
    user=st.secrets["database"]["username"],
    password=st.secrets["database"]["password"]
)
''', language="python")

st.warning("⚠️ Always add `.streamlit/secrets.toml` to `.gitignore`!")

---

## 🎯 Practical Example: Student Database Dashboard

Build a complete database-backed dashboard.

In [ ]:
import streamlit as st
import sqlite3
import pandas as pd

st.header("🎯 Practical: Student Dashboard")

# --- Database Functions ---
@st.cache_resource
def get_connection():
    """Get cached database connection."""
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            score REAL,
            grade TEXT
        )
    """)
    # Seed data
    conn.executemany(
        "INSERT INTO students (name, score, grade) VALUES (?, ?, ?)",
        [
            ("Alice", 95, "A"),
            ("Bob", 87, "B+"),
            ("Charlie", 92, "A-"),
            ("Diana", 78, "C+"),
            ("Eve", 96, "A+"),
        ]
    )
    conn.commit()
    return conn

@st.cache_data(ttl=10)
def get_students():
    """Get all students."""
    conn = get_connection()
    return pd.read_sql("SELECT * FROM students", conn)

def add_student(name, score, grade):
    """Add a new student."""
    conn = get_connection()
    conn.execute(
        "INSERT INTO students (name, score, grade) VALUES (?, ?, ?)",
        (name, score, grade)
    )
    conn.commit()
    get_students.clear()

# --- UI ---
df = get_students()

# Metrics
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Total Students", len(df))
with col2:
    st.metric("Average Score", f"{df['score'].mean():.1f}")
with col3:
    st.metric("Highest Score", f"{df['score'].max()}")

# Add student
with st.expander("➕ Add New Student"):
    with st.form("add"):
        name = st.text_input("Name")
        score = st.number_input("Score", 0, 100, 85)
        grade = st.selectbox("Grade", ["A+", "A", "A-", "B+", "B", "B-", "C+", "C"])
        if st.form_submit_button("Add"):
            add_student(name, score, grade)
            st.success(f"Added {name}!")
            st.rerun()

# Display
st.dataframe(df, use_container_width=True)
st.bar_chart(df.set_index("name")["score"])

---

## ⚠️ Common Errors & Debugging

### Error 1: Database is Locked
```python
# Symptom: sqlite3.OperationalError: database is locked
# Fix: Use check_same_thread=False and WAL mode
conn = sqlite3.connect("data.db", check_same_thread=False)
conn.execute("PRAGMA journal_mode=WAL")
```

### Error 2: Table Doesn't Exist
```python
# Symptom: sqlite3.OperationalError: no such table
# Fix: Initialize database schema on first run
def init_db():
    conn.execute("CREATE TABLE IF NOT EXISTS ...")
```

### Error 3: Stale Data
```python
# Symptom: Old data displayed after writes
# Fix: Clear cache after modifications
def add_record():
    conn.execute("INSERT INTO ...")
    get_data.clear()  # Invalidate cache
```

---

## 🎯 Challenges

### Challenge 1: Product Inventory
Build a product inventory with:
- Add/edit/delete products
- Search by name
- Filter by category
- Display statistics

### Challenge 2: Todo List
Create a todo app with:
- Add tasks with priority
- Mark as complete
- Delete tasks
- Filter by status

### Challenge 3: Grade Book
Build a grade book with:
- Students and assignments
- Record grades
- Calculate averages
- Export report

In [ ]:
# Challenge 1: Product Inventory
import streamlit as st
import sqlite3
import pandas as pd

st.write("TODO: Build a product inventory database app")

# Your code here


---

## 📝 Key Takeaways

1. **Databases provide** persistent, queryable, concurrent data storage

2. **SQLite is ideal for learning** — no setup, built into Python

3. **Always use parameterized queries** — never concatenate user input

4. **Cache connections with `@st.cache_resource`** — singletons

5. **Cache query results with `@st.cache_data`** — copies

6. **Clear cache after writes** — `get_data.clear()`

7. **Manage secrets securely** — use `st.secrets`, never hardcode

---

## 📚 Further Reading

- [SQLite Python Docs](https://docs.python.org/3/library/sqlite3.html)
- [Streamlit Secrets](https://docs.streamlit.io/develop/concepts/connections/secrets-management)

---

## 🔗 Related Materials

- 📖 Reading: [14 — Databases & Persistence](../readings/14_databases_and_persistence.md)
- ✏️ Exercise: [14 — Database Workshop](../exercises/14_database_workshop.py)
- 🖥️ Demo App: [14 — Database Dashboard](../apps/14_database_dashboard.py)
- 📝 Quiz: [10 — Databases & Persistence](../quizzes/10_databases_persistence.md)